## Q1. Design a Cognitive Assistant for Reading Labels of Medicine Bottle

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
!pip -q install pytesseract opencv-python-headless

In [ ]:
import cv2, re, io
import numpy as np
import pytesseract
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

uploaded = files.upload()
image = Image.open(io.BytesIO(next(iter(uploaded.values())))).convert("RGB")
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.show()

image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
medicine_text = pytesseract.image_to_string(gray, config="--psm 6")

print("Extracted label text:")
print(medicine_text)
lines = [line.strip() for line in medicine_text.splitlines() if line.strip()]
strengths = re.findall(r"\b\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|mL|%|IU)\b", medicine_text, re.I)
date_strings = re.findall(r"\b(?:0?[1-9]|1[0-2])[/-](?:\d{2}|\d{4})\b|\b\d{2}[/-]\d{2}[/-]\d{2,4}\b", medicine_text)
print("Possible medicine name:", lines[0] if lines else "Not detected")
print("Possible strength/quantity:", strengths)
print("Possible date strings:", date_strings)
print("Manually verify every field. OCR can misread medicine names, dosage, and expiry dates.")

## Q2. Design an Emotion-aware Cognitive Assistant using Facial Expression Detection

In [ ]:
!pip -q install deepface tf-keras

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from deepface import DeepFace

uploaded_face = files.upload()
face_image = Image.open(io.BytesIO(next(iter(uploaded_face.values())))).convert("RGB")
plt.figure(figsize=(6, 6))
plt.imshow(face_image)
plt.axis("off")
plt.show()

result = DeepFace.analyze(
    img_path=np.array(face_image),
    actions=["emotion"],
    enforce_detection=False,
    detector_backend="opencv"
)
result = result[0] if isinstance(result, list) else result
print("Predicted expression category:", result.get("dominant_emotion", "Not detected"))
print("Expression scores:", result.get("emotion", {}))